<a href="https://colab.research.google.com/github/mih36-boop/Nasij/blob/main/nlp/Nasij_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U sentence-transformers scikit-learn pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 114.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [2]:
!pip install -q pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 101.5 MB/s eta 0:00:00


In [3]:
import pandas as pd
print(pd.__version__)

2.2.2


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [5]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Model loaded successfully!")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
data = {
    "suggestion": [

        # Waste management
        "Garbage has not been collected for several days.",
        "We need more garbage bins around Hamra.",
        "Trash is piling up on the sidewalks.",
        "الزبالة عم تتراكم بالشارع وما حدا عم يشيلها",
        "بدنا حاويات زبالة أكتر بالمنطقة",
        "لازم البلدية تجمع النفايات بشكل أسرع",

        # Roads / potholes
        "The potholes on our street are dangerous.",
        "Please repair the damaged roads in the neighborhood.",
        "The road needs urgent maintenance.",
        "الحفر بالطرقات صارت كتير خطيرة",
        "لازم يصلحوا الطريق لأنه كله حفر",
        "الطرقات بحاجة لصيانة بأسرع وقت",

        # Street lighting
        "Several streetlights are not working at night.",
        "Our street needs better lighting.",
        "Please repair the broken street lamps.",
        "ضو الشارع مش شغال بالليل",
        "بدنا إنارة أحسن بالمنطقة",
        "في عواميد إنارة كتير ما عم تشتغل",

        # Green spaces
        "We need more trees and green spaces.",
        "Please create more public parks in the neighborhood.",
        "There should be more greenery around residential areas.",
        "بدنا شجر أكتر بالمنطقة",
        "لازم يكون في مساحات خضرا أكتر",
        "بدنا حدائق عامة أكتر للناس",

        # Water supply
        "We have frequent water shortages.",
        "The neighborhood needs a more reliable water supply.",
        "Water cuts are happening too often.",
        "المي عم تنقطع كتير بالمنطقة",
        "بدنا حل لمشكلة انقطاع المي",
        "توزيع المي بالحي مش منتظم"
    ],

    "true_topic": [
        "waste", "waste", "waste", "waste", "waste", "waste",
        "roads", "roads", "roads", "roads", "roads", "roads",
        "lighting", "lighting", "lighting", "lighting", "lighting", "lighting",
        "green_spaces", "green_spaces", "green_spaces",
        "green_spaces", "green_spaces", "green_spaces",
        "water", "water", "water", "water", "water", "water"
    ]
}

df = pd.DataFrame(data)

df

,suggestion,true_topic
0,Garbage has not been collected for several days.,waste
1,We need more garbage bins around Hamra.,waste
2,Trash is piling up on the sidewalks.,waste
3,الزبالة عم تتراكم بالشارع وما حدا عم يشيلها,waste
4,بدنا حاويات زبالة أكتر بالمنطقة,waste
5,لازم البلدية تجمع النفايات بشكل أسرع,waste
6,The potholes on our street are dangerous.,roads
7,Please repair the damaged roads in the neighbo...,roads
8,The road needs urgent maintenance.,roads
9,الحفر بالطرقات صارت كتير خطيرة,roads


In [7]:
print("Number of suggestions:", len(df))

print("\nSuggestions per topic:")
print(df["true_topic"].value_counts())

Number of suggestions: 30

Suggestions per topic:
true_topic
waste           6
roads           6
lighting        6
green_spaces    6
water           6
Name: count, dtype: int64


In [8]:
embeddings = model.encode(df["suggestion"].tolist())

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (30, 384)


In [9]:
print("Original sentence:")
print(df["suggestion"][0])

print("\nFirst 10 numbers of its embedding:")
print(embeddings[0][:10])

Original sentence:
Garbage has not been collected for several days.

First 10 numbers of its embedding:
[-0.05692208  0.0291448   0.22821938 -0.19102003  0.23324351 -0.19551322
 -0.13037632 -0.0351497  -0.06162111 -0.16575706]


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
sentence1 = "Garbage has not been collected for several days."
sentence2 = "Trash is piling up on the sidewalks."

embedding1 = model.encode([sentence1])
embedding2 = model.encode([sentence2])

similarity = cosine_similarity(embedding1, embedding2)[0][0]

print("Similarity:", similarity)

Similarity: 0.49105588


In [12]:
sentence1 = "The potholes on our street are dangerous."
sentence2 = "الحفر بالطرقات صارت كتير خطيرة"

embedding1 = model.encode([sentence1])
embedding2 = model.encode([sentence2])

similarity = cosine_similarity(embedding1, embedding2)[0][0]

print("English-Arabic similarity:", similarity)

English-Arabic similarity: 0.7961493


In [13]:
sentence1 = "Garbage has not been collected for several days."
sentence2 = "We need more trees and green spaces."

embedding1 = model.encode([sentence1])
embedding2 = model.encode([sentence2])

similarity = cosine_similarity(embedding1, embedding2)[0][0]

print("Unrelated similarity:", similarity)

Unrelated similarity: 0.06319892


In [14]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(embeddings)

df["cluster"] = clusters

df[["suggestion", "true_topic", "cluster"]]

,suggestion,true_topic,cluster
0,Garbage has not been collected for several days.,waste,2
1,We need more garbage bins around Hamra.,waste,2
2,Trash is piling up on the sidewalks.,waste,2
3,الزبالة عم تتراكم بالشارع وما حدا عم يشيلها,waste,0
4,بدنا حاويات زبالة أكتر بالمنطقة,waste,0
5,لازم البلدية تجمع النفايات بشكل أسرع,waste,2
6,The potholes on our street are dangerous.,roads,3
7,Please repair the damaged roads in the neighbo...,roads,3
8,The road needs urgent maintenance.,roads,3
9,الحفر بالطرقات صارت كتير خطيرة,roads,3


In [15]:
comparison = pd.crosstab(
    df["true_topic"],
    df["cluster"]
)

comparison

cluster,0,1,2,3,4
true_topic,,,,,
green_spaces,1,5,0,0,0
lighting,3,0,0,0,3
roads,1,0,0,5,0
waste,2,0,4,0,0
water,3,0,3,0,0


In [16]:
score = silhouette_score(embeddings, clusters)

print("Silhouette Score:", score)

Silhouette Score: 0.14076420664787292


In [17]:
embeddings_normalized = model.encode(
    df["suggestion"].tolist(),
    normalize_embeddings=True
)

print("Shape:", embeddings_normalized.shape)

Shape: (30, 384)


In [18]:
kmeans_normalized = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

clusters_normalized = kmeans_normalized.fit_predict(
    embeddings_normalized
)

df["cluster_normalized"] = clusters_normalized

In [19]:
comparison_normalized = pd.crosstab(
    df["true_topic"],
    df["cluster_normalized"]
)

comparison_normalized

cluster_normalized,0,1,2,3,4
true_topic,,,,,
green_spaces,0,0,5,1,0
lighting,0,0,0,0,6
roads,6,0,0,0,0
waste,0,4,0,1,1
water,0,3,0,3,0


In [20]:
score_normalized = silhouette_score(
    embeddings_normalized,
    clusters_normalized
)

print("Original Silhouette Score:", score)
print("Normalized Silhouette Score:", score_normalized)

Original Silhouette Score: 0.14076420664787292
Normalized Silhouette Score: 0.11170809715986252


In [21]:
from sklearn.cluster import AgglomerativeClustering

In [22]:
agg = AgglomerativeClustering(
    n_clusters=5,
    metric="cosine",
    linkage="average"
)

clusters_agg = agg.fit_predict(embeddings)

df["cluster_agg"] = clusters_agg

In [23]:
pd.crosstab(
    df["true_topic"],
    df["cluster_agg"]
)

cluster_agg,0,1,2,3,4
true_topic,,,,,
green_spaces,0,0,0,5,1
lighting,0,4,0,1,1
roads,0,6,0,0,0
waste,4,1,0,0,1
water,2,0,2,1,1


In [24]:
score_agg = silhouette_score(
    embeddings,
    clusters_agg,
    metric="cosine"
)

print("K-Means Silhouette:", score)
print("Agglomerative Silhouette:", score_agg)

K-Means Silhouette: 0.14076420664787292
Agglomerative Silhouette: 0.19528864324092865


In [25]:
#evaluate k-means using cosine too
score_kmeans_cosine = silhouette_score(
    embeddings,
    clusters,
    metric="cosine"
)

print("K-Means Cosine Silhouette:", score_kmeans_cosine)
print("Agglomerative Cosine Silhouette:", score_agg)

K-Means Cosine Silhouette: 0.20182247459888458
Agglomerative Cosine Silhouette: 0.19528864324092865


In [26]:
#agglomerative crosstab
pd.crosstab(
    df["true_topic"],
    df["cluster_agg"]
)

cluster_agg,0,1,2,3,4
true_topic,,,,,
green_spaces,0,0,0,5,1
lighting,0,4,0,1,1
roads,0,6,0,0,0
waste,4,1,0,0,1
water,2,0,2,1,1


In [27]:
#calculating the ARI
from sklearn.metrics import adjusted_rand_score

In [28]:
ari_kmeans = adjusted_rand_score(
    df["true_topic"],
    df["cluster"]
)

ari_agg = adjusted_rand_score(
    df["true_topic"],
    df["cluster_agg"]
)

print("K-Means ARI:", ari_kmeans)
print("Agglomerative ARI:", ari_agg)

K-Means ARI: 0.35488877392653906
Agglomerative ARI: 0.31756254644538023


In [ ]:
#from now on i use df["cluster"] not df["cluster_agg"] bc it's closer to 1
#which means that it's K-means is closer to perfect topic grouping

In [29]:
#need to inspect what each cluster is about and label it
for cluster_id in sorted(df["cluster"].unique()):
    print(f"\nCLUSTER {cluster_id}")
    print("-" * 50)

    cluster_rows = df[df["cluster"] == cluster_id]

    for suggestion in cluster_rows["suggestion"]:
        print("-", suggestion)


CLUSTER 0
--------------------------------------------------
- الزبالة عم تتراكم بالشارع وما حدا عم يشيلها
- بدنا حاويات زبالة أكتر بالمنطقة
- لازم يصلحوا الطريق لأنه كله حفر
- ضو الشارع مش شغال بالليل
- بدنا إنارة أحسن بالمنطقة
- في عواميد إنارة كتير ما عم تشتغل
- بدنا شجر أكتر بالمنطقة
- المي عم تنقطع كتير بالمنطقة
- بدنا حل لمشكلة انقطاع المي
- توزيع المي بالحي مش منتظم

CLUSTER 1
--------------------------------------------------
- We need more trees and green spaces.
- Please create more public parks in the neighborhood.
- There should be more greenery around residential areas.
- لازم يكون في مساحات خضرا أكتر
- بدنا حدائق عامة أكتر للناس

CLUSTER 2
--------------------------------------------------
- Garbage has not been collected for several days.
- We need more garbage bins around Hamra.
- Trash is piling up on the sidewalks.
- لازم البلدية تجمع النفايات بشكل أسرع
- We have frequent water shortages.
- The neighborhood needs a more reliable water supply.
- Water cuts are happeni

In [30]:
#bc cluster 0 for example contains mostly arabic suggestions from many different topics
#we'll try multilingual-e5-small to see if it's a better model
better_model = SentenceTransformer(
    "intfloat/multilingual-e5-small"
)

print("New model loaded!")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

New model loaded!


In [31]:
#generate new embeddings
e5_embeddings = better_model.encode(
    df["suggestion"].tolist(),
    normalize_embeddings=True
)

print(e5_embeddings.shape)

(30, 384)


In [32]:
#run k-means
kmeans_e5 = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

clusters_e5 = kmeans_e5.fit_predict(e5_embeddings)

df["cluster_e5"] = clusters_e5

In [33]:
pd.crosstab(
    df["true_topic"],
    df["cluster_e5"]
)

cluster_e5,0,1,2,3,4
true_topic,,,,,
green_spaces,3,0,0,0,3
lighting,3,3,0,0,0
roads,0,1,5,0,0
waste,2,0,2,1,1
water,3,0,2,0,1


In [34]:
score_e5 = silhouette_score(
    e5_embeddings,
    clusters_e5,
    metric="cosine"
)

ari_e5 = adjusted_rand_score(
    df["true_topic"],
    df["cluster_e5"]
)

print("Old K-Means Cosine Silhouette:", score_kmeans_cosine)
print("E5 Cosine Silhouette:", score_e5)

print("\nOld K-Means ARI:", ari_kmeans)
print("E5 ARI:", ari_e5)

Old K-Means Cosine Silhouette: 0.20182247459888458
E5 Cosine Silhouette: 0.1640757918357849

Old K-Means ARI: 0.35488877392653906
E5 ARI: 0.13165399239543726


In [ ]:
#e5 is worse for our data set so i'll keep paraphrase-multilingual-MiniLM-L12-v2
# + k-means

In [35]:
#creating the 5 topic prototypes
topic_descriptions = {
    "Waste management":
        "garbage collection, trash, waste, garbage bins and dirty streets",

    "Roads / potholes":
        "potholes, damaged roads, road repairs and street maintenance",

    "Street lighting":
        "broken streetlights, dark streets, street lamps and public lighting",

    "Green spaces":
        "trees, parks, gardens, greenery and public green spaces",

    "Water supply":
        "water shortages, water cuts, water distribution and reliable water supply"
}

In [36]:
#turn them into embeddings
topic_names = list(topic_descriptions.keys())
topic_texts = list(topic_descriptions.values())

topic_embeddings = model.encode(
    topic_texts,
    normalize_embeddings=True
)

print("Topic embeddings shape:", topic_embeddings.shape)

Topic embeddings shape: (5, 384)


In [37]:
#classify the 30 suggestions
suggestion_embeddings = model.encode(
    df["suggestion"].tolist(),
    normalize_embeddings=True
)

similarities = cosine_similarity(
    suggestion_embeddings,
    topic_embeddings
)

best_topic_indices = similarities.argmax(axis=1)

df["semantic_topic"] = [
    topic_names[i] for i in best_topic_indices
]

df[["suggestion", "true_topic", "semantic_topic"]]

,suggestion,true_topic,semantic_topic
0,Garbage has not been collected for several days.,waste,Waste management
1,We need more garbage bins around Hamra.,waste,Waste management
2,Trash is piling up on the sidewalks.,waste,Waste management
3,الزبالة عم تتراكم بالشارع وما حدا عم يشيلها,waste,Waste management
4,بدنا حاويات زبالة أكتر بالمنطقة,waste,Waste management
5,لازم البلدية تجمع النفايات بشكل أسرع,waste,Waste management
6,The potholes on our street are dangerous.,roads,Roads / potholes
7,Please repair the damaged roads in the neighbo...,roads,Roads / potholes
8,The road needs urgent maintenance.,roads,Roads / potholes
9,الحفر بالطرقات صارت كتير خطيرة,roads,Roads / potholes


In [38]:
#did it fix our arabic problem?
pd.crosstab(
    df["true_topic"],
    df["semantic_topic"]
)

semantic_topic,Green spaces,Roads / potholes,Street lighting,Waste management,Water supply
true_topic,,,,,
green_spaces,6,0,0,0,0
lighting,1,0,5,0,0
roads,0,6,0,0,0
waste,0,0,0,6,0
water,0,0,1,2,3


In [39]:
topic_mapping = {
    "waste": "Waste management",
    "roads": "Roads / potholes",
    "lighting": "Street lighting",
    "green_spaces": "Green spaces",
    "water": "Water supply"
}

df["true_topic_name"] = df["true_topic"].map(topic_mapping)

accuracy = (
    df["true_topic_name"] == df["semantic_topic"]
).mean()

print(f"Semantic topic accuracy: {accuracy:.2%}")

Semantic topic accuracy: 86.67%


In [40]:
topic_descriptions_bilingual = {
    "Waste management":
        "garbage collection, trash, waste, garbage bins, dirty streets, "
        "جمع النفايات، الزبالة، حاويات الزبالة، النفايات في الشوارع",

    "Roads / potholes":
        "potholes, damaged roads, road repairs, street maintenance, "
        "الحفر، الطرقات المتضررة، تصليح الطرق، صيانة الطرقات",

    "Street lighting":
        "broken streetlights, dark streets, street lamps, public lighting, "
        "إنارة الشوارع، عواميد الإنارة، ضو الشارع، إنارة لا تعمل",

    "Green spaces":
        "trees, parks, gardens, greenery, public green spaces, "
        "شجر، حدائق، مساحات خضراء، حدائق عامة",

    "Water supply":
        "water shortages, water cuts, water distribution, reliable water supply, "
        "انقطاع المي، انقطاع المياه، توزيع المياه، نقص المياه"
}

In [41]:
topic_names = list(topic_descriptions_bilingual.keys())
topic_texts = list(topic_descriptions_bilingual.values())

topic_embeddings_bilingual = model.encode(
    topic_texts,
    normalize_embeddings=True
)

similarities_bilingual = cosine_similarity(
    suggestion_embeddings,
    topic_embeddings_bilingual
)

best_indices = similarities_bilingual.argmax(axis=1)

df["semantic_topic_bilingual"] = [
    topic_names[i] for i in best_indices
]

In [42]:
pd.crosstab(
    df["true_topic"],
    df["semantic_topic_bilingual"]
)

semantic_topic_bilingual,Green spaces,Roads / potholes,Street lighting,Waste management,Water supply
true_topic,,,,,
green_spaces,6,0,0,0,0
lighting,1,0,5,0,0
roads,0,6,0,0,0
waste,0,1,0,5,0
water,0,0,2,1,3


In [43]:
accuracy_bilingual = (
    df["true_topic_name"] == df["semantic_topic_bilingual"]
).mean()

print(f"Original prototype accuracy: {accuracy:.2%}")
print(f"Bilingual prototype accuracy: {accuracy_bilingual:.2%}")

Original prototype accuracy: 86.67%
Bilingual prototype accuracy: 83.33%


In [ ]:
#tge bilingual prototype is worse

In [44]:
df["final_topic"] = df["semantic_topic"]

print("Final NLP method: Semantic topic assignment")
print(f"Accuracy: {accuracy:.2%}")

Final NLP method: Semantic topic assignment
Accuracy: 86.67%


In [45]:
#identifying the most requested topic
topic_counts = df["final_topic"].value_counts()

print("Citizen concerns ranked by frequency:\n")
print(topic_counts)

Citizen concerns ranked by frequency:

final_topic
Waste management    8
Green spaces        7
Roads / potholes    6
Street lighting     6
Water supply        3
Name: count, dtype: int64


In [46]:
#to explicitlu see the mistakes
mistakes = df[
    df["true_topic_name"] != df["final_topic"]
]

mistakes[
    ["suggestion", "true_topic_name", "final_topic"]
]

,suggestion,true_topic_name,final_topic
16,بدنا إنارة أحسن بالمنطقة,Street lighting,Green spaces
27,المي عم تنقطع كتير بالمنطقة,Water supply,Waste management
28,بدنا حل لمشكلة انقطاع المي,Water supply,Street lighting
29,توزيع المي بالحي مش منتظم,Water supply,Waste management


In [47]:
incoming_suggestions = [
    "The potholes near our building are getting worse.",
    "Please fix the damaged road near the university.",
    "There are huge holes in the road.",
    "الحفر بالطريق عم تسبب مشاكل للسيارات",
    "الطريق بحاجة لتصليح سريع",
    "Our road has been damaged for months.",

    "Garbage is piling up next to our building.",
    "We need more garbage bins.",
    "The municipality needs to collect trash more often.",
    "الزبالة بالشارع صارت كتير",
    "بدنا حاويات نفايات أكتر",

    "The water has been cut for two days.",
    "المي عم تنقطع كل يوم",
    "We need a more reliable water supply.",

    "Several street lamps are not working.",
    "الشارع عتمة بالليل لأن الضو مش شغال",

    "We need more trees in our neighborhood.",
    "Please create a small public park."
]

incoming_df = pd.DataFrame({
    "suggestion": incoming_suggestions
})

incoming_df

,suggestion
0,The potholes near our building are getting worse.
1,Please fix the damaged road near the university.
2,There are huge holes in the road.
3,الحفر بالطريق عم تسبب مشاكل للسيارات
4,الطريق بحاجة لتصليح سريع
5,Our road has been damaged for months.
6,Garbage is piling up next to our building.
7,We need more garbage bins.
8,The municipality needs to collect trash more o...
9,الزبالة بالشارع صارت كتير


In [48]:
#classify the new citizen submissions
incoming_embeddings = model.encode(
    incoming_df["suggestion"].tolist(),
    normalize_embeddings=True
)

incoming_similarities = cosine_similarity(
    incoming_embeddings,
    topic_embeddings
)

incoming_best_indices = incoming_similarities.argmax(axis=1)

incoming_df["topic"] = [
    topic_names[i] for i in incoming_best_indices
]

incoming_df

,suggestion,topic
0,The potholes near our building are getting worse.,Roads / potholes
1,Please fix the damaged road near the university.,Roads / potholes
2,There are huge holes in the road.,Roads / potholes
3,الحفر بالطريق عم تسبب مشاكل للسيارات,Roads / potholes
4,الطريق بحاجة لتصليح سريع,Roads / potholes
5,Our road has been damaged for months.,Roads / potholes
6,Garbage is piling up next to our building.,Waste management
7,We need more garbage bins.,Waste management
8,The municipality needs to collect trash more o...,Waste management
9,الزبالة بالشارع صارت كتير,Street lighting


In [49]:
incoming_counts = incoming_df["topic"].value_counts()

print("Most requested citizen concerns:\n")
print(incoming_counts)

Most requested citizen concerns:

topic
Roads / potholes    6
Waste management    5
Street lighting     3
Water supply        2
Green spaces        2
Name: count, dtype: int64


In [50]:
print("TOP PRIORITY:")
print(incoming_counts.index[0])

print("\nNumber of suggestions:")
print(incoming_counts.iloc[0])

TOP PRIORITY:
Roads / potholes

Number of suggestions:
6


In [51]:
topic_summaries = {}

for topic in incoming_counts.index:

    # Get all citizen suggestions assigned to this topic
    topic_rows = incoming_df[incoming_df["topic"] == topic]
    topic_suggestions = topic_rows["suggestion"].tolist()

    # Embed those suggestions
    topic_suggestion_embeddings = model.encode(
        topic_suggestions,
        normalize_embeddings=True
    )

    # Get the prototype embedding for this topic
    topic_index = topic_names.index(topic)
    prototype_embedding = topic_embeddings[topic_index].reshape(1, -1)

    # Compare every suggestion with the topic prototype
    scores = cosine_similarity(
        topic_suggestion_embeddings,
        prototype_embedding
    ).flatten()

    # Select the most representative suggestion
    best_index = scores.argmax()
    representative = topic_suggestions[best_index]

    topic_summaries[topic] = representative

In [56]:
for topic, count in incoming_counts.items():

    representative = topic_summaries[topic]

    print(f"\n{topic}:")

    print(f"Number of citizen suggestions: {count}")
    print(f"Representative concern: {representative}")


Roads / potholes:
Number of citizen suggestions: 6
Representative concern: There are huge holes in the road.

Waste management:
Number of citizen suggestions: 5
Representative concern: Garbage is piling up next to our building.

Street lighting:
Number of citizen suggestions: 3
Representative concern: Several street lamps are not working.

Water supply:
Number of citizen suggestions: 2
Representative concern: We need a more reliable water supply.

Green spaces:
Number of citizen suggestions: 2
Representative concern: Please create a small public park.


In [55]:
#final municipality summary
print("NASIJ: CITIZEN FEEDBACK SUMMARY")


for rank, (topic, count) in enumerate(incoming_counts.items(), start=1):

    representative = topic_summaries[topic]

    print(f"\n#{rank} {topic}")
    print(f"Requests: {count}")
    print(f"Summary: Citizens are raising concerns related to {topic.lower()}.")
    print(f"Representative feedback: {representative}")

NASIJ: CITIZEN FEEDBACK SUMMARY

#1 Roads / potholes
Requests: 6
Summary: Citizens are raising concerns related to roads / potholes.
Representative feedback: There are huge holes in the road.

#2 Waste management
Requests: 5
Summary: Citizens are raising concerns related to waste management.
Representative feedback: Garbage is piling up next to our building.

#3 Street lighting
Requests: 3
Summary: Citizens are raising concerns related to street lighting.
Representative feedback: Several street lamps are not working.

#4 Water supply
Requests: 2
Summary: Citizens are raising concerns related to water supply.
Representative feedback: We need a more reliable water supply.

#5 Green spaces
Requests: 2
Summary: Citizens are raising concerns related to green spaces.
Representative feedback: Please create a small public park.
